In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import json
import torch
from collections import OrderedDict, defaultdict
from diffusers import StableDiffusionPipeline

os.environ["CUDA_VISIBLE_DEVICES"] = '1'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
pipe = pipe.to(DEVICE)

unet = pipe.unet
unet.eval()

pipe.safety_checker = None # 안전 필터 off
# pipe.set_progress_bar_config(disable=True) # 내부 tqdm 출력 off

In [3]:
def get_target_block(unet, target_block_name: str):
    parts = target_block_name.split(".")
    def resolve(obj, remaining):
        if not remaining:
            return obj
        part, *rest = remaining
        child = obj[int(part)] if part.isdigit() else getattr(obj, part)
        return resolve(child, rest)
    return resolve(unet, parts)

In [5]:
ff = get_target_block(pipe.unet, "up_blocks.1.attentions.1.transformer_blocks.0.ff")
ff_state = ff.state_dict()

In [7]:
ff_state.keys()

odict_keys(['net.0.proj.weight', 'net.0.proj.bias', 'net.2.weight', 'net.2.bias'])

In [12]:
W_geglu = ff_state["net.0.proj.weight"]   # (10240, 1280)
W_out   = ff_state["net.2.weight"]        # (1280, 5120)
W_geglu.shape, W_out.shape

(torch.Size([10240, 1280]), torch.Size([1280, 5120]))

In [ ]:
n_geglu_half = W_geglu.shape[0] // 2 # 5120

W2 = W_geglu[n_geglu_half:]
b_geglu = ff_state["net.0.proj.bias"]
b2 = b_geglu[n_geglu_half:]

In [ ]:
j = 0
w_j = W2[j, :]
b_j = b2[j]

# signed_dist = ( @ w_j + b_j) / w_j.norm()